#### Setup

Let's set up our environment and define a couple of useful functions.
We'll first import the OpenAI client, and then a few additional Python libraries we'll use to parse the function calls and execute our specific function code.

For this exercise, we'll offer one simple function to the API that it can call if it seems appropriate: `roll_dice`, a random number generator which offers arguments for both number of sides and number of dice

In [1]:
from openai import OpenAI
import json
import random

client = OpenAI()

# Define functions that we'll expose to the LLM
def roll_dice(sides=6, count=1):
    """Simulates rolling dice with the specified number of sides and count."""
    results = [random.randint(1, sides) for _ in range(count)]
    return {
        "rolls": results,
        "total": sum(results)
    }


The first step in function calling is defining our tools for the LLM. This involves creating a schema that describes our function, its purpose, and the parameters it accepts.

These tool definitions are then included in each prompt we send to the model API.

We're using OpenAI, which defines a particular schema it expects for function calls. This may vary if you use other language model providers, but the core concepts should remain the same.

Using OpenAI, we'll send over a list of `tools`, where each tool is a dictionary with the format specified below. The `"parameters"` of the function are defined by a JSON Schema, which you can learn more about at https://json-schema.org/.

Read the dictionary below carefully to understand the format OpenAI expects.

```python
{
        "type": "function",
        "function": {
            "name": "name_of_this_function",
            "description": "A description of what the function accepts and returns. Note that this is 'read' by the LLM and is a form of prompting! Make sure your explanation is clear and thorough so the LLM knows what to do.",
            "parameters": {
                "type": "object", # the parameters must *always* be of type 'object'
                "properties": {
                    "a_string": {
                        "type": "string",
                        "description": "Describe what information this string should contain here!"
                    },
                    "an_integer": {
                        "type": "integer",
                        "description": "Again, describe the purpose of the integer in the function here, remembering that it's read by the model."
                    }
                },
                "required": [], # here you can pass the names of arguments that should be required by the function
                "additionalProperties": False # False means the LLM can only use the properties explicitly defined in the schema; True would allow addiitional properties to be read
            },
            "strict": True # this setting is specific to OpenAI's SDK, and tries to force the LLM to adhere strictly to the schema format, reducing errors
        }
    }
```

#### Checkpoint 1/4

Now fill in the code below to define the `roll_dice` tool for the OpenAI API. The function should accept two properties:

- `sides`, an integer specifying the number of sides on each die
- `count`, an integer specifying how many dice to roll

Neither are required. `strict` should be set to `True`.

Do your best to make the descriptions of both the function and each property as detailed as possible. Feel free to look at the function definition in the setup cell to refresh yourself on what the function does.

In [2]:
dice_tool_definition = {
  "type": "function",
  "function": {
    "name": "roll_dice",
    "description": "Simulates rolling dice with the specified number of sides and count.",
    "parameters": {
      "type": "object",
      "properties": {
        "sides": {"type": "integer", "description": "Number of sides on each die."},
        "count": {"type": "integer", "description": "Number of dice to roll."}
      }
    },
    "required": [],
    "additionalProperties": False
  },
  "strict": True
}

print(json.dumps(dice_tool_definition, indent=2))

{
  "type": "function",
  "function": {
    "name": "roll_dice",
    "description": "Simulates rolling dice with the specified number of sides and count.",
    "parameters": {
      "type": "object",
      "properties": {
        "sides": {
          "type": "integer",
          "description": "Number of sides on each die."
        },
        "count": {
          "type": "integer",
          "description": "Number of dice to roll."
        }
      }
    },
    "required": [],
    "additionalProperties": false
  },
  "strict": true
}


Next, we should test the function out by sending a prompt to the OpenAI API with a request that should trigger it.

Typically, the OpenAI Chat Completions API will send a regular text response back with a `message.content` containing the assistant's output text.

However when the OpenAI Chat Completions API receives a prompt that indicates it needs to call a function, its response will contain a `message.tool_calls` list, with the `content` value optional. Each tool call will be a dictionary of the following shape:

```python
[
    {
        "id": "call_9876abc",
        "type": "function",
        "function": {
            "name": "function_name",
            "arguments": "{\"key\":\"value",}"
        }
    },
]
```

We'll now define a helper function, `process_tool_calls`, that will take the message from the OpenAI API response and pull out any tool calls it contains.

Below, we first extract the `message` from the response. Then if the message does not contain a `tool_calls` value, we return `None`.

If it *does* contain a call to one of our tools, we extract the first tool call in the list (we're assuming it only calls one tool at a time.)

We then extract the name of this function and parse its arguments into a dictionary, returning the tool call's `"id"`, as well as its name and arguments, into a dict.

This helper function will make it easy to grab and execute tool calls whenever they come back from the API.

In [3]:
def process_tool_calls(response):
    """
    Extract function call details from the API response.
    """
    message = response.choices[0].message
    
    if not hasattr(message, "tool_calls") or not message.tool_calls:
        return None
    
    # Get the first tool call
    tool_call = message.tool_calls[0]
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)
    return {
        "id": tool_call.id,
        "function_name": function_name,
        "function_args": function_args
    }



#### Checkpoint 2/4

Now that we've defined our tool and its specifications, and prepared to handle tool calls returning from the OpenAI API, we can set up our prompts to the model to contain information about the tools it has at its disposal.

We've created the `prompt_with_tools` function to assist us in this task.

Your first task will be to define `tools` as a list of tool definitions. In this exercise, there should only be one item in this list: the dice rolling schema we defined in the previous checkpoint.

Then, pass the tool definition list to the `client.chat.completions.create()` method, passing your tools list to the `tools` parameter. We return the `messages` list in addition to the response for the sake of code we'll write later in this exercise.

Then, execute the code cell. You'll see `process_tool_calls` working its magic, with the function call parsed and printed to the console.

If you want to confirm that the tool call doesn't register if the prompt doesn't seem to need one, pass "What is the capital of France?" to the `"content"` key in the model prompt instead.


In [4]:
def prompt_with_tools(prompt):
    tools = [dice_tool_definition]

    messages = [{"role": "user", "content": prompt}]
    # Send a request to the model
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=messages,
        tools=tools
    )
    return response, messages

response, _ = prompt_with_tools("Roll two six-sided dice for me.")

# Process the response
tool_call = process_tool_calls(response)
if tool_call:
    print("Tool call ID:", tool_call["id"])
    print("Function name:", tool_call["function_name"])
    print("Function arguments:", tool_call["function_args"])
else:
    print("No function call in the response")

Tool call ID: call_LQKKWocL6OyQQWN1VKpGg9Rf
Function name: roll_dice
Function arguments: {'sides': 6, 'count': 2}


Great! We've now defined a function ('tool'), prompted the LLM while offering it, and received a function call from the LLM in response.

All that's left is to receive the function call, parse and call it, and send the results back to the LLM for the final completion.



#### Checkpoint 3/4

Our next step: write a function that will parse the LLM's tool call, extract its id and the function's name and arguments, execute the appropriate function code, and return the result.

First, pull the three values out of the dictionary returned by `process_tool_call`. Then, extract from `function_args` the two arguments and pass them as named arguments to the `roll_dice` function. Note that since `roll_dice` is our only function, we'll return an error if the LLM calls anything else.

When you're done, execute the code cell and see what we'll pass to the model in the next checkpoint.


In [5]:
def function_executor(tool_call):
    tool_call_id = tool_call["id"]
    function_name = tool_call["function_name"]
    function_args = tool_call["function_args"]

    if function_name == "roll_dice":
        sides = function_args['sides']
        count = function_args['count']
        result = roll_dice(sides=sides, count=count)
    else:
        result = {"error": f"Unknown function: {function_name}"}
    
    print(f"Function result: {result}")
    return tool_call_id, result

function_executor(tool_call)

Function result: {'rolls': [2, 5], 'total': 7}


('call_LQKKWocL6OyQQWN1VKpGg9Rf', {'rolls': [2, 5], 'total': 7})

#### Checkpoint 4/4
Alright, it's time to send our result back to the language model and obtain our first full LLM completion with tool calling.

We've set up `complete_function_workflow` with calls to the functions we've already defined thus far in this exercise. Now let's send our function's results back, along with the history of the interaction, when we prompt the model one last time. This updated `messages` list—which contains the original prompt, the model's decision to call the function, and our results—will be sent to the LLM one final time, where the results will be incorporated into its output.

We first send the model's tool call and the results of our execution of that tool back to the API. Note we include the tool call's `id` when we pass along the `result` of our function calling. Finally, we prompt the model using the completed message history, extract the result, and return it.

We've already appended the model's decision to call the tool to the `messages` list. Your task will be to append the tool call's results to the messages list immediately afterward.

- As usual, the message will be a dict
- The `"role"` of the message is `"tool"`
- The `"tool_call_id"` should be passed as a key, with the actual id as its value
- The `"content"` should be a JSONified version of the function execution result (convert the result dict to JSON with `json.dumps(result)`)

Then at the bottom of the cell, prompt `complete_function_workflow` with the `dice_prompt` and enjoy seeing the model execute code on your request!


In [6]:

def complete_function_workflow(prompt):
    """Complete end-to-end function calling workflow."""
    response, messages = prompt_with_tools(prompt)

    tool_call = process_tool_calls(response)
    if not tool_call:
        return response.choices[0].message.content
    
    tool_call_id, result = function_executor(tool_call)
    
    messages.append(response.choices[0].message)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call_id,
        "content": json.dumps(result)
    })

    final_response = client.chat.completions.create(
        model="gpt-4.1",
        messages=messages
    )
    
    return final_response.choices[0].message.content

dice_prompt = "Roll two six-sided dice and then tell me a fun fact about the numbers rolled."
dice_response = complete_function_workflow(dice_prompt)
print(f"Final response: {dice_response}\n")

Function result: {'rolls': [1, 4], 'total': 5}
Final response: You rolled a 1 and a 4!

Fun fact: 1 and 4 are both perfect squares (1 = 1×1, 4 = 2×2). But here's something even cooler: If you put them together, “14” is the atomic number of silicon, the main element used to make computer chips!

